In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, f1_score
from train_loop import train_loop

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=3)
        self.maxpool = nn.MaxPool2d(2)
        self.fc = nn.Linear(675, 2)

    def forward(self, x):
        x = torch.relu(self.conv(x))
        x = self.maxpool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [ ]:
# For CIFAR-10 binary classification (e.g., cats vs dogs)
transform_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

full_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)
# Filter and remap labels in one go
class RemappedSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, class_map):
        self.dataset = dataset
        self.class_map = class_map
        self.indices = [i for i, (_, label) in enumerate(dataset) if label in class_map]

    def __len__(self):
        return len(self.indices)//5
        # return 800

    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        return img, self.class_map[label]

# Use it
class_map = {3: 0, 5: 1}   # cat -> 0, dog -> 1
train_dataset = RemappedSubset(full_train, class_map)
# # Filter for two classes (e.g., class 3=cat, class 5=dog)
# train_indices = [i for i, (_, label) in enumerate(full_train) if label in [3, 5]]
# train_dataset = Subset(full_train, train_indices)

# Create DataLoader
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

In [ ]:
len(train_loader.dataset)

In [ ]:
xshape = train_dataset.dataset[0][0].shape
model = Net()
loss_fn = nn.CrossEntropyLoss()

opt = torch_numopt.NewtonLS(
    model=model,
    solver="pinv",
    line_search_method="backtrack",
    line_search_cond="armijo",
    damping="identity",
    mu=1e-2
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    train_loader,
    epochs=1000,
    max_patience=10
)